In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random

# Read data
df_raw = pd.read_csv('manga.csv')

# Select relevant columns for the recommendation system
selected_cols = ['Title', 'Score', 'Vote', 'Ranked', 'Members', 'Genres', 'Themes', 'Author', 'Demographics']
df = df_raw[selected_cols].copy()

In [2]:
# Validate Members column and clean it
df['Members'] = df['Members'].astype(str).str.replace(',', '', regex=False).str.strip()
df['Members'] = pd.to_numeric(df['Members'], errors='coerce').fillna(0).astype(int)

# Clean text data in Genres, Themes, Author, Demographics
text_features = ['Genres', 'Themes', 'Author', 'Demographics']
for col in text_features:
    df[col] = df[col].astype(str).replace(['[]', 'nan', 'None'], '', regex=False)
    df[col] = df[col].str.replace(r"[\[\]']", "", regex=True).str.lower().str.strip()

# Remove entries where both Genres and Themes are empty, as they provide no content information
df = df[~((df['Genres'] == "") & (df['Themes'] == ""))]
df = df.reset_index(drop=True) 

# Create content soup by combining Genres, Themes, Author, and Demographics
df['content_soup'] = (df['Genres'] + ' ') * 2 + df['Themes'] + ' ' + df['Author'] + ' ' + df['Demographics']
df['content_soup'] = df['content_soup'].str.replace(r'\s+', ' ', regex=True).str.strip()

In [3]:
# Vectorize content_soup using TF-IDF
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df['content_soup'])

# Calculate cosine similarity between manga based on their content vectors
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a Series to map manga titles to their indices in the DataFrame
indices = pd.Series(df.index, index=df['Title']).drop_duplicates()

In [4]:
def get_recommendations(title, k=10):
    if title not in indices:
        return "Không tìm thấy truyện trong hệ thống."
    
    # In case of duplicate titles, take the first one
    idx = indices[title]
    if hasattr(idx, "__len__"): idx = idx[0]
    
    # Calculate similarity scores for all manga with the given title
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Top 50 candidates based on content similarity
    candidate_indices = [i[0] for i in sim_scores[1:51]]
    candidates = df.iloc[candidate_indices].copy()
    
    # Rerank candidates by Score and Members to prioritize popular and highly rated manga
    final_recs = candidates.sort_values(by=['Score', 'Members'], ascending=False)
    
    return final_recs.head(k)

In [5]:
def evaluate_coverage(title, k=10):
    recs = get_recommendations(title, k=k)
    if isinstance(recs, str): return None
    
    orig_genres = set(df[df['Title'] == title]['Genres'].iloc[0].split())
    if not orig_genres: return 0.0
    
    scores = []
    for _, row in recs.iterrows():
        rec_genres = set(row['Genres'].split())
        intersection = orig_genres.intersection(rec_genres)
        union = orig_genres.union(rec_genres)
        scores.append(len(intersection) / len(union) if union else 0)
        
    return np.mean(scores)

In [21]:
# Take a random sample of 50 manga titles to evaluate the system's content coverage
sample_titles = random.sample(df['Title'].tolist(), 50)
all_scores = []

for title in sample_titles:
    s = evaluate_coverage(title)
    if s is not None:
        all_scores.append(s)

mean_system_score = sum(all_scores) / len(all_scores)
print(f"Mean Content Coverage của toàn hệ thống: {mean_system_score:.2%}")

Mean Content Coverage của toàn hệ thống: 70.34%


C:\Users\Admin\AppData\Local\Temp\ipykernel_7484\3473414967.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if hasattr(idx, "__len__"): idx = idx[0]


In [7]:
# Test the recommendation system with a specific manga title
test_name = "One Piece" # Hoặc Naruto
print(f"Độ phủ nội dung cho {test_name}: {evaluate_coverage(test_name):.2%}")
print(get_recommendations(test_name, k=5))

Độ phủ nội dung cho One Piece: 93.50%
                                             Title  Score    Vote  Ranked  \
39                                 Hunter x Hunter   8.74  122355      43   
288                                     D.Gray-man   8.28   62648     325   
310                                Saiyuuki Gaiden   8.26    2879     348   
483  One Piece: Episode A (One Piece: Ace's Story)   8.12    4921     543   
542                        One Piece: Strong World   8.08    8803     611   

     Members                      Genres Themes  \
39    265723  action, adventure, fantasy          
288   156291  action, adventure, fantasy          
310    10622  action, adventure, fantasy          
483     9721  action, adventure, fantasy          
542    15639  action, adventure, fantasy          

                                                Author Demographics  \
39                    togashi, yoshihiro (story & art)      shounen   
288                     hoshino, katsura (story & 